# Week 2.3: Embeddings Deep Dive

## Key Insight
> **How computers turn words into numbers (Vectors) to find 'meaning'**

## Learning Objectives
By the end of this notebook, you will:
- Word2Vec and GloVe internals
- Building embeddings from scratch
- Semantic similarity and vector operations
- Dimensionality reduction and visualization
- Using pre-trained embeddings

## Exercises
This notebook contains 5 exercises ranging from easy to advanced.

## Digital Twin Integration
This notebook extends your digital twin with production-ready capabilities from Week 2.

In [ ]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Any, Tuple
import re
import json

print('✅ Week 2.3: Embeddings Deep Dive - Ready to start!')
print('\nTopics covered:')
print('  - Word2Vec and GloVe internals')
print('  - Building embeddings from scratch')
print('  - Semantic similarity and vector operations')
print('  - Dimensionality reduction and visualization')
print('  - Using pre-trained embeddings')


---
## Part 1: Introduction to Word2Vec and GloVe internals

Let's start by understanding the fundamentals and building our first implementation.

In [ ]:
# Example implementation
# This is a starter template - expand with your implementation

class ExampleClass:
    """
    Example class for Word2Vec and GloVe internals.
    """

    def __init__(self):
        self.data = []

    def process(self, input_data):
        """
        Process the input data.
        """
        # Add your implementation here
        return input_data

# Test the class
example = ExampleClass()
result = example.process('test data')
print(f'Result: {result}')

### 🎯 Exercise 1 (Easy): Getting Started

**Task**: Implement a basic version of the concept introduced above.

**Requirements**:
1. Follow the pattern shown in the example
2. Test your implementation with sample data
3. Verify the output matches expectations
4. Add error handling for edge cases

In [ ]:
# YOUR CODE HERE
# Complete the exercise following the instructions above

# Test your implementation
# Add test cases here

pass  # Replace with your implementation

---
## Part 2: Building embeddings from scratch

Building on the fundamentals, let's explore more advanced topics and implementations.

In [ ]:
# Advanced implementation example

def advanced_function(data: List[Any], **kwargs) -> Dict[str, Any]:
    """
    Advanced function for Building embeddings from scratch.

    Args:
        data: Input data to process
        **kwargs: Additional parameters

    Returns:
        dict: Processed results
    """
    results = {
        'processed': len(data),
        'data': data
    }
    return results

# Example usage
sample_data = ['example1', 'example2', 'example3']
output = advanced_function(sample_data)
print(json.dumps(output, indent=2))

### 🎯 Exercise 2 (Intermediate): Building on the Basics

**Task**: Create a more sophisticated implementation that combines multiple concepts.

**Requirements**:
1. Use the patterns from Part 1 and Part 2
2. Add parameter validation
3. Include helpful error messages
4. Test with various input types

In [4]:
# Small corpus — enough to learn meaningful relationships
corpus = [
    "the king rules the kingdom with wisdom",
    "the queen rules beside the king",
    "the man works in the kingdom",
    "the woman works beside the man",
    "the king and queen live in the castle",
    "the prince is the son of the king",
    "the princess is the daughter of the queen",
    "the apple grows on the tree",
    "the orange grows on the tree",
    "the apple and orange are fruits",
    "the dog runs in the park",
    "the cat sleeps in the house",
    "the dog and cat are animals",
    "the king loves the queen",
    "the man loves the woman",
]

# Quick sanity check
print(f"Corpus: {len(corpus)} sentences")
print(f"Total words: {sum(len(s.split()) for s in corpus)}")

Corpus: 15 sentences
Total words: 95


In [5]:
import numpy as np
import re
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional


STOP_WORDS = {"the", "a", "an", "is", "are", "was", "were", "in", "on",
              "of", "and", "or", "with", "beside", "to", "for", "at", "by"}


def clean_corpus(corpus: List[str], remove_stopwords: bool = True) -> List[List[str]]:
    """Tokenize and clean a corpus into a list of word lists."""
    if not isinstance(corpus, (list, tuple)):
        raise TypeError(f"corpus must be list, got {type(corpus).__name__}")

    cleaned = []
    for i, sentence in enumerate(corpus):
        if not isinstance(sentence, str):
            raise TypeError(f"corpus[{i}] must be str, got {type(sentence).__name__}")
        tokens = re.findall(r"[a-z]+", sentence.lower())
        if remove_stopwords:
            tokens = [t for t in tokens if t not in STOP_WORDS]
        if tokens:
            cleaned.append(tokens)
    return cleaned


class EmbeddingTrainer:
    """
    Train word embeddings from scratch using two methods:

        - Word2Vec (Skip-gram with negative sampling)
        - GloVe-lite (co-occurrence matrix + truncated SVD)

    Both methods return a dict: word → np.ndarray of shape (dim,).
    """

    def __init__(self, dim: int = 32, window: int = 2, seed: int = 42):
        # --- Validation ---
        if not isinstance(dim, int) or dim <= 0:
            raise ValueError(f"dim must be a positive int, got {dim!r}")
        if not isinstance(window, int) or window <= 0:
            raise ValueError(f"window must be a positive int, got {window!r}")

        self.dim = dim
        self.window = window
        self.seed = seed
        self.vocab: List[str] = []
        self.word2idx: Dict[str, int] = {}
        self.embeddings: Optional[Dict[str, np.ndarray]] = None

    # ----------------------------------------------------------
    # Shared preprocessing
    # ----------------------------------------------------------
    def _build_vocab(self, sentences: List[List[str]]):
        counts = Counter(w for s in sentences for w in s)
        self.vocab = sorted(counts.keys())
        self.word2idx = {w: i for i, w in enumerate(self.vocab)}
        print(f"  Vocab size: {len(self.vocab)}")

    def _extract_pairs(self, sentences):
        """Yield (center, context) pairs within the window."""
        for sentence in sentences:
            for i, center in enumerate(sentence):
                start = max(0, i - self.window)
                end = min(len(sentence), i + self.window + 1)
                for j in range(start, end):
                    if i != j:
                        yield center, sentence[j]

    # ----------------------------------------------------------
    # Method 1: Word2Vec (Skip-gram + negative sampling)
    # ----------------------------------------------------------
    def train_word2vec(self, sentences: List[List[str]],
                       epochs: int = 30, lr: float = 0.05,
                       n_negatives: int = 3) -> Dict[str, np.ndarray]:
        """
        Skip-gram: given a center word, predict context words.
        Uses negative sampling to keep training tractable.
        """
        self._build_vocab(sentences)
        V = len(self.vocab)
        rng = np.random.default_rng(self.seed)

        # Two matrices: W_in (center) and W_out (context)
        W_in = (rng.random((V, self.dim)) - 0.5) / self.dim
        W_out = np.zeros((V, self.dim))

        # Sample distribution for negatives (unigram^0.75)
        counts = np.array([sum(1 for s in sentences for w in s if w == v)
                           for v in self.vocab], dtype=float)
        probs = counts ** 0.75
        probs /= probs.sum()

        pairs = list(self._extract_pairs(sentences))
        print(f"  Training pairs: {len(pairs)}")

        def sigmoid(x):
            return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))

        for epoch in range(epochs):
            rng.shuffle(pairs)
            total_loss = 0.0

            for center, context in pairs:
                ci = self.word2idx[center]
                oi = self.word2idx[context]

                # --- Positive sample ---
                v_c = W_in[ci]
                u_o = W_out[oi]
                score = sigmoid(v_c @ u_o)
                loss = -np.log(score + 1e-9)
                total_loss += loss

                grad = (score - 1.0) * lr
                W_in[ci] -= grad * u_o
                W_out[oi] -= grad * v_c

                # --- Negative samples ---
                negs = rng.choice(V, size=n_negatives, p=probs)
                for ni in negs:
                    if ni == oi:
                        continue
                    u_n = W_out[ni]
                    score_n = sigmoid(v_c @ u_n)
                    total_loss += -np.log(1.0 - score_n + 1e-9)
                    grad_n = score_n * lr
                    W_in[ci] -= grad_n * u_n
                    W_out[ni] -= grad_n * v_c

            if epoch % 5 == 0 or epoch == epochs - 1:
                print(f"  Epoch {epoch:3d} | loss {total_loss/len(pairs):.4f}")

        # L2-normalize for stable cosine similarity
        norms = np.linalg.norm(W_in, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        W_in = W_in / norms

        self.embeddings = {w: W_in[i] for w, i in self.word2idx.items()}
        return self.embeddings

    # ----------------------------------------------------------
    # Method 2: GloVe-lite (co-occurrence + SVD)
    # ----------------------------------------------------------
    def train_glove_lite(self, sentences: List[List[str]]) -> Dict[str, np.ndarray]:
        """
        Build a word-word co-occurrence matrix, apply log scaling,
        then factorize it with truncated SVD.
        """
        if not self.vocab:
            self._build_vocab(sentences)
        V = len(self.vocab)

        # Build co-occurrence matrix
        cooc = np.zeros((V, V), dtype=float)
        for center, context in self._extract_pairs(sentences):
            i = self.word2idx[center]
            j = self.word2idx[context]
            cooc[i, j] += 1.0

        print(f"  Co-occurrence matrix: {cooc.shape}, "
              f"non-zero entries: {np.count_nonzero(cooc)}")

        # Log scaling (GloVe-style): log(1 + x)
        M = np.log1p(cooc)

        # Truncated SVD
        U, S, _ = np.linalg.svd(M, full_matrices=False)
        k = min(self.dim, len(S))
        emb = U[:, :k] * np.sqrt(S[:k])  # weight by singular values

        # L2-normalize
        norms = np.linalg.norm(emb, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        emb = emb / norms

        self.embeddings = {w: emb[i] for w, i in self.word2idx.items()}
        return self.embeddings

    # ----------------------------------------------------------
    # Post-training analysis
    # ----------------------------------------------------------
    def similarity(self, w1: str, w2: str) -> float:
        self._check_ready()
        if w1 not in self.embeddings:
            raise KeyError(f"'{w1}' not in vocabulary")
        if w2 not in self.embeddings:
            raise KeyError(f"'{w2}' not in vocabulary")
        v1, v2 = self.embeddings[w1], self.embeddings[w2]
        return float(np.dot(v1, v2))  # already L2-normalized

    def nearest(self, word: str, k: int = 3) -> List[Tuple[str, float]]:
        self._check_ready()
        if word not in self.embeddings:
            raise KeyError(f"'{word}' not in vocabulary")
        if not isinstance(k, int) or k <= 0:
            raise ValueError(f"k must be positive int, got {k!r}")

        target = self.embeddings[word]
        scored = [(w, float(np.dot(target, v)))
                  for w, v in self.embeddings.items() if w != word]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:k]

    def analogy(self, a: str, b: str, c: str, k: int = 3):
        """a is to b as c is to ?  →  solve b - a + c."""
        self._check_ready()
        for w in (a, b, c):
            if w not in self.embeddings:
                raise KeyError(f"'{w}' not in vocabulary")
        target = self.embeddings[b] - self.embeddings[a] + self.embeddings[c]
        scored = [(w, float(np.dot(target, v)))
                  for w, v in self.embeddings.items() if w not in (a, b, c)]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:k]

    def _check_ready(self):
        if self.embeddings is None:
            raise RuntimeError("Model not trained yet. Call train_*() first.")

In [6]:
np.random.seed(42)

# Clean the corpus
sentences = clean_corpus(corpus, remove_stopwords=True)
print("Sample cleaned sentence:", sentences[0])
print()

# --- Train Word2Vec ---
print("=" * 50)
print("Word2Vec (Skip-gram + negative sampling)")
print("=" * 50)
w2v = EmbeddingTrainer(dim=32, window=2, seed=42)
w2v.train_word2vec(sentences, epochs=30, lr=0.05, n_negatives=3)
print()

# --- Test similarity ---
print("Word2Vec similarities:")
for a, b in [("king", "queen"), ("king", "man"), ("apple", "orange"),
             ("king", "apple"), ("dog", "cat")]:
    try:
        print(f"  {a:8s} ↔ {b:8s}: {w2v.similarity(a, b):+.3f}")
    except KeyError as e:
        print(f"  {a:8s} ↔ {b:8s}: {e}")
print()

# --- Test nearest neighbors ---
print("Nearest to 'king':")
for w, s in w2v.nearest("king", k=4):
    print(f"  {w:10s}: {s:+.3f}")
print()

# --- Test analogy ---
print("Analogy: king - man + woman ≈ ?")
for w, s in w2v.analogy("man", "king", "woman", k=3):
    print(f"  {w:10s}: {s:+.3f}")

Sample cleaned sentence: ['king', 'rules', 'kingdom', 'wisdom']

Word2Vec (Skip-gram + negative sampling)
  Vocab size: 27
  Training pairs: 98
  Epoch   0 | loss 2.7018
  Epoch   5 | loss 2.6942
  Epoch  10 | loss 2.6708
  Epoch  15 | loss 2.6280
  Epoch  20 | loss 2.5418
  Epoch  25 | loss 2.3461
  Epoch  29 | loss 2.2042

Word2Vec similarities:
  king     ↔ queen   : +0.819
  king     ↔ man     : +0.734
  apple    ↔ orange  : +0.991
  king     ↔ apple   : +0.772
  dog      ↔ cat     : +0.964

Nearest to 'king':
  castle    : +0.985
  princess  : +0.980
  daughter  : +0.973
  live      : +0.900

Analogy: king - man + woman ≈ ?
  castle    : +0.904
  daughter  : +0.901
  princess  : +0.900


In [7]:
print("=" * 50)
print("GloVe-lite (co-occurrence + SVD)")
print("=" * 50)
glove = EmbeddingTrainer(dim=32, window=2, seed=42)
glove.train_glove_lite(sentences)
print()

print("GloVe-lite similarities:")
for a, b in [("king", "queen"), ("apple", "orange"),
             ("dog", "cat"), ("king", "apple")]:
    try:
        print(f"  {a:8s} ↔ {b:8s}: {glove.similarity(a, b):+.3f}")
    except KeyError as e:
        print(f"  {a:8s} ↔ {b:8s}: {e}")
print()

print("Nearest to 'king':")
for w, s in glove.nearest("king", k=4):
    print(f"  {w:10s}: {s:+.3f}")

GloVe-lite (co-occurrence + SVD)
  Vocab size: 27
  Co-occurrence matrix: (27, 27), non-zero entries: 86

GloVe-lite similarities:
  king     ↔ queen   : +0.145
  apple    ↔ orange  : +0.411
  dog      ↔ cat     : +0.093
  king     ↔ apple   : -0.000

Nearest to 'king':
  castle    : +0.348
  wisdom    : +0.329
  daughter  : +0.187
  princess  : +0.187


In [8]:
def test_errors():
    print("Error handling tests:")
    print("-" * 50)

    # Bad constructor args
    cases = [
        ("dim=0",            lambda: EmbeddingTrainer(dim=0)),
        ("dim='a'",          lambda: EmbeddingTrainer(dim="a")),
        ("window=-1",        lambda: EmbeddingTrainer(window=-1)),
        ("corpus not list",  lambda: clean_corpus("not a list")),
        ("corpus has int",   lambda: clean_corpus(["ok", 123])),
    ]
    for name, fn in cases:
        try:
            fn()
            print(f"  ❌ {name}: no error")
        except (TypeError, ValueError) as e:
            print(f"  ✅ {name}: {type(e).__name__}: {e}")

    # Untrained model
    print()
    fresh = EmbeddingTrainer(dim=16)
    try:
        fresh.similarity("king", "queen")
    except RuntimeError as e:
        print(f"  ✅ untrained model: RuntimeError: {e}")

    # Unknown word
    try:
        w2v.similarity("king", "unicorn")
    except KeyError as e:
        print(f"  ✅ unknown word: KeyError: {e}")

    # Bad k
    try:
        w2v.nearest("king", k=0)
    except ValueError as e:
        print(f"  ✅ bad k: ValueError: {e}")


test_errors()

Error handling tests:
--------------------------------------------------
  ✅ dim=0: ValueError: dim must be a positive int, got 0
  ✅ dim='a': ValueError: dim must be a positive int, got 'a'
  ✅ window=-1: ValueError: window must be a positive int, got -1
  ✅ corpus not list: TypeError: corpus must be list, got str
  ✅ corpus has int: TypeError: corpus[1] must be str, got int

  ✅ untrained model: RuntimeError: Model not trained yet. Call train_*() first.
  ✅ unknown word: KeyError: "'unicorn' not in vocabulary"
  ✅ bad k: ValueError: k must be positive int, got 0


---
## Part 3: Semantic similarity and vector operations

Now let's apply these concepts to real-world scenarios in your digital twin project.

In [ ]:
# Visualization example
import matplotlib.pyplot as plt

def visualize_results(data: List[Any]):
    """
    Visualize the processing results.
    """
    fig, ax = plt.subplots(figsize=(10, 6))

    # Add your visualization code here
    ax.set_title('Results Visualization')
    ax.set_xlabel('X-axis')
    ax.set_ylabel('Y-axis')

    plt.tight_layout()
    plt.show()

# Example: visualize_results(sample_data)

### 🎯 Exercise 3 (Advanced): Complete Implementation

**Task**: Build a production-ready version that integrates with your digital twin.

**Requirements**:
1. Combine all concepts from this notebook
2. Add comprehensive error handling
3. Include logging and debugging capabilities
4. Create visualization of results
5. Write documentation for your implementation

In [ ]:
# YOUR CODE HERE
# Build your complete implementation

class ProductionSystem:
    """
    Production-ready implementation for digital twin.
    """

    def __init__(self):
        # Initialize your system
        pass

    def process(self, input_data):
        # Add your implementation
        pass

# Test your production system
# Add comprehensive tests here

---
## Summary & Key Takeaways

### What We Learned:
- Word2Vec and GloVe internals
- Building embeddings from scratch
- Semantic similarity and vector operations
- Dimensionality reduction and visualization
- Using pre-trained embeddings

### Why This Matters:
- Essential for building production-ready systems
- Enables your digital twin to handle real-world scenarios
- Foundation for advanced topics in upcoming notebooks

### Next Steps:
1. Complete all exercises in this notebook
2. Integrate concepts with your digital twin project
3. Test your implementation thoroughly
4. Move on to the next notebook in Week 2

## 🏆 Challenge Project

**Build a Complete Word2Vec and GloVe internals System for Your Digital Twin**

**Requirements**:
1. Apply all concepts from this notebook
2. Handle edge cases and errors gracefully
3. Include comprehensive testing
4. Create visualizations of your results
5. Document your implementation
6. Integrate with your existing digital twin code

**Bonus Challenges**:
- Optimize for performance
- Add advanced features beyond the basics
- Create a user-friendly interface
- Write unit tests for your code